# Uploading Content

Upload video, audio, or image files to Jockey as assets. This notebook covers both direct (local file) and URL-based upload methods.

In [ ]:
import json
import os
import time

import requests

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
BASE_URL = "https://api.twelvelabs.io/v1.3"
HEADERS = {"x-api-key": API_KEY, "Content-Type": "application/json"}

## When You Need This

Before you can build a knowledge store, you need content. This guide covers both upload methods:

- **Direct Upload** — local files under 200MB
- **URL Upload** — remote files up to 4GB

## Helper: Wait for Asset Processing

Assets are processed asynchronously after upload. This helper polls until the asset reaches `ready` or `failed` status.

In [ ]:
def wait_for_asset_ready(
    asset_id: str,
    headers: dict,
    base_url: str = BASE_URL,
    interval: int = 5,
    timeout: int = 600,
) -> dict:
    """Poll an asset until it reaches 'ready' or 'failed' status.

    Args:
        asset_id: The ID of the asset to monitor.
        headers: Request headers including the API key.
        base_url: The base URL for the Jockey API.
        interval: Seconds between polling attempts.
        timeout: Maximum seconds to wait before raising an error.

    Returns:
        The asset response dict once it reaches 'ready' status.

    Raises:
        Exception: If the asset fails processing or the timeout is exceeded.
    """
    elapsed = 0
    while elapsed < timeout:
        response = requests.get(f"{base_url}/assets/{asset_id}", headers=headers)
        asset_data = response.json()
        status = asset_data["status"]

        if status == "ready":
            print(f"Asset {asset_id} is ready.")
            return asset_data
        elif status == "failed":
            raise Exception(f"Asset {asset_id} processing failed.")

        print(f"Asset status: {status} (elapsed: {elapsed}s)")
        time.sleep(interval)
        elapsed += interval

    raise Exception(f"Timeout after {timeout}s waiting for asset {asset_id}.")

## Direct Upload (Local File)

Best for files on your machine under 200MB. Note that for direct uploads, we use `data` and `files` parameters instead of `json`, and we omit the `Content-Type` header so `requests` can set the multipart boundary automatically.

In [ ]:
# Direct upload — local file
UPLOAD_HEADERS = {"x-api-key": API_KEY}  # No Content-Type for multipart uploads

LOCAL_FILE_PATH = "video.mp4"  # Replace with your file path

with open(LOCAL_FILE_PATH, "rb") as f:
    response = requests.post(
        f"{BASE_URL}/assets",
        headers=UPLOAD_HEADERS,
        data={"method": "direct"},
        files={"file": f},
    )

asset = response.json()
print(f"Asset ID: {asset['_id']}, Status: {asset['status']}")

## URL Upload (Remote File)

Best for files already hosted online, or files larger than 200MB (up to 4GB). The URL must be publicly accessible — Jockey fetches the file directly.

In [ ]:
# URL upload — remote file
REMOTE_VIDEO_URL = "https://example.com/large-video.mp4"  # Replace with your URL

response = requests.post(
    f"{BASE_URL}/assets",
    headers=UPLOAD_HEADERS,
    data={
        "method": "url",
        "url": REMOTE_VIDEO_URL,
    },
)

asset = response.json()
print(f"Asset ID: {asset['_id']}, Status: {asset['status']}")

## Waiting for Processing

Assets are processed asynchronously. You must wait for the asset to reach `ready` status before adding it to a knowledge store.

In [ ]:
# Wait for the uploaded asset to be ready
asset_id = asset["_id"]
ready_asset = wait_for_asset_ready(asset_id, {"x-api-key": API_KEY})
print(json.dumps(ready_asset, indent=2))

## Optional Features: HLS Streaming and Thumbnails

Enable HLS streaming and thumbnail generation at upload time:

In [ ]:
# Upload with HLS streaming and thumbnail generation enabled
response = requests.post(
    f"{BASE_URL}/assets",
    headers=UPLOAD_HEADERS,
    data={
        "method": "url",
        "url": "https://example.com/video.mp4",
        "enable_hls": "true",
        "enable_thumbnail": "true",
    },
)

asset_with_extras = response.json()
print(f"Asset ID: {asset_with_extras['_id']}, Status: {asset_with_extras['status']}")

## Common Pitfalls

- **File too large for direct upload** — Use the URL method for files over 200MB.
- **URL not accessible** — Jockey must be able to fetch the URL. Check that it is publicly reachable.
- **Forgetting to wait** — An asset must be `ready` before you can add it to a knowledge store. Always poll for status.

## Next Steps

- [Building Knowledge Stores](building_knowledge_stores.ipynb) — organize uploaded assets into queryable collections
- [Ingestion Config](ingestion_config.ipynb) — control what Jockey extracts from your videos
- [Querying](querying.ipynb) — ask questions about your video collection

**API Reference:** [POST /assets](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/upload-content/direct-uploads/create)